**Loading the data**

We load the ground truth data generated previously.

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("./ground_truth-mal-2.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

Load the documents and build a minsearch index:

In [2]:
from ingest import load_data, build_index_keyword

documents = load_data()

index = build_index_keyword(documents)

### Search Evaluation

Now that we have ground truth data, we can evaluate how well our search retrieves the correct documents.

For each question in our ground truth dataset, we run search. Then we check whether the results include the correct document.

#### Setting up search (text Search)

We'll use the ground truth CSV from the previous lesson and set up the search index here.

For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.

Load the ground truth file from Generating Ground truth notebook:

##### Testing search text

Wrap the search call in a function called text_search. The name is deliberate. 

Later we'll write vector_search or a hybrid version and run the exact same evaluation on it. 

Everything downstream only needs a function that takes a query and returns results, so we can swap one for another. 

That mirrors how RAG works: the retrieval step doesn't care which search function sits behind it.

In [3]:
def text_search(query):
    boost_dict = {'synopsis': 3.0, 'title_english': 1, 'studios': 1.0, 'genres': 1.0, 'source': 1.0}
    #filter_dict = {''} 
    
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
        #filter_dict=filter_dict
)

##### Testing Collecting relevance data (text search query only)

Start with one ground truth record:

In [40]:
q = ground_truth[0]
q

{'question': 'Can anyone recommend an anime about an immortal elf mage reflecting on a long-ago hero’s journey and trying to understand human relationships better?',
 'document': 52991}

Run search for this question:

In [41]:
doc_id = q['document']
results = text_search(query=q['question'])
results

[{'mal_id': 7791,
  'title': 'K-On!!',
  'title_english': 'K-ON! Season 2',
  'type': 'TV',
  'source': '4-koma manga',
  'episodes': 26.0,
  'status': 'Finished Airing',
  'airing': False,
  'rating': 'PG-13 - Teens 13 or older',
  'score': 8.18,
  'scored_by': 411363,
  'rank': 458.0,
  'popularity': 305,
  'members': 748024,
  'favorites': 15234,
  'synopsis': 'It is the new year, which means that the senior members of the Light Music Club are now third-years, with Azusa Nakano being the only second-year. The seniors soon realize that Azusa will be the only member left once they graduate and decide to recruit new members. Despite trying many methods of attracting underclassmen—handing out fliers, bringing people into the clubroom, and performing at the welcoming ceremony—there are no signs of anyone that plans to join.\n\nWhile heading to the clubroom, Azusa overhears Yui Hirasawa say that the club is fine with only five people and that they can do many fun things together. Changing

First, compare the retrieved document IDs with the correct document ID:

In [42]:
for d in results:
    print(f'{d['mal_id']} == {doc_id}: {d['mal_id'] == doc_id}')

7791 == 52991: False
49918 == 52991: False
9617 == 52991: False
59978 == 52991: False
53447 == 52991: False


Then turn this comparison into a relevance list. In this lesson, relevance means whether a retrieved document is the correct document for this question.

In [43]:
#testing relevance in one record
relevance = []

for d in results:
    relevance.append(int(d["mal_id"] == doc_id))

relevance

[0, 0, 0, 0, 0]

This gives a list of 0 and 1 values. 1 means the retrieved document has the same ID as the correct document.

Put this logic into a relevance function that will calculate relevance:

In [44]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["mal_id"] == doc_id))

    return relevance

For the first ground truth record, the relevance list is:

In [45]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)


Can anyone recommend an anime about an immortal elf mage reflecting on a long-ago hero’s journey and trying to understand human relationships better?


[0, 0, 0, 0, 0]

This gives a list of 0 and 1 values. 1 means the retrieved document has the same ID as the correct document.


### Compute relevance for all ground truth questions:

In [4]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["mal_id"] == doc_id))

    return relevance

In [5]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

#### Testing compute relevance

Call it for the first 15 ground truth questions:

In [4]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [47]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

Look at the results:

In [15]:
relevance_total_text

[[0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

Each entry in relevance_total_text is a relevance list. This is enough to check that the function works before we run it for the full dataset.

Next, **make the relevance functions generic**. We start with text search, but later we may want to evaluate vector search, hybrid search, or another retrieval method. The relevance logic is the same. Only the search function changes.

In [48]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["mal_id"] == doc_id))

    return relevance

The total relevance function gets a search_function too.

We need to provide it explicitly:

In [49]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

Use it with text_search on the same sample:

In [50]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0]]

Now run it for all ground truth questions:

In [51]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/2500 [00:00<?, ?it/s]

Now we can represent search results as relevance lists. In the next lesson, we'll turn these lists into metrics: Hit Rate and MRR.

### Search Evaluation Metrics

In the previous section, we computed relevance lists for search results. We can turn those lists into metrics.


#### Hit Rate

Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:

In [6]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

##### Testing Hit Rate

In [52]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

Each line is one query. If a line contains 1, search found the correct document somewhere in the top 5 results. 

If the line contains only zeros, search did not find the correct document.

In our setup, each query has one correct document, so Hit Rate and Recall@k mean the same thing.

Let's calculate it:

In [53]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

There are 14 hits. The example has 15 queries.

The Hit Rate is:

In [54]:
cnt / len(example)
# 0.933

0.9333333333333333

This means that search found the correct document for 93.3% of the queries in this example.

### Mean Reciprocal Rank (MRR)

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct document:

* position 1: score is 1.0
* position 2: score is 0.5
* position 3: score is 0.333
* not found: score is 0

In the example, most hits are at the first position. Some hits are lower in the list.

Look at that line:

In [7]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

##### Testing Mean Reciprocal Rank (MRR)

In [57]:
example[1]
# [0, 1, 0, 0, 0]

[0, 1, 0, 0, 0]

For this line, the score is 1/2 because the correct document is at position 2.

Let's calculate MRR:

In [58]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

The total score is 12.333333333333334. 

We use rank + 1 because Python counts positions from zero. 

The first position should score 1/1, and without the + 1 we'd divide by zero.

Divide it by the number of queries:

In [59]:
total_score / len(example)
# 0.822

0.8222222222222222

MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.

Hit Rate is the upper bound for MRR. In practice, MRR is usually smaller because some correct documents are found below the first position.

Put the same logic into a function:

Check it on the same example:

In [61]:
mrr(example)
# 0.822

0.8222222222222222

### WORKING WITH ALL METRICS AND EVALUATIONS TOGETHER

Wrap the metrics in a reusable evaluation function:

In [8]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

We can evaluate any search function:

In [9]:
evaluate(
    ground_truth, #No slice evaluates all dataset
    text_search
)

  0%|          | 0/2500 [00:00<?, ?it/s]

{'hit_rate': 0.5132, 'mrr': 0.34375333333333463}

You should see something like:

In [32]:
{"hit_rate": 0.899, "mrr": 0.769}

{'hit_rate': 0.899, 'mrr': 0.769}

Search metrics tell us whether retrieval works. Next, we'll use these metrics to tune the search parameters.

#### Interpreting the metrics

A few things to keep in mind when reading these numbers:

Our ground truth assumes only one relevant document per query. In practice, other retrieved documents might also be relevant. 

A 50% hit rate does not mean that half the results are useless. 

It means the document we generated the question from did not appear in the top results for half the queries. 

Other relevant documents may still be there.

With synthetic data, the generated questions can be too close to the original FAQ text. 

This inflates hit rate and MRR. If you see numbers above 95%, treat them with caution and check whether the questions are realistic enough.

Good thresholds depend on your use case. A 50% hit rate is acceptable for some applications, while others need 90% or higher. The right number depends on how much the downstream LLM can compensate for imperfect retrieval. It also depends on user tolerance for wrong answers.

Look at the system holistically. A high MRR means the relevant document is near the top, which helps the LLM focus on the right context. 

A low MRR with a high hit rate means the document is there, but buried under less relevant results.


### Search Parameter Tuning

In the previous lesson, we defined Hit Rate, MRR, and the evaluate function. Now we can use them to tune search parameters.

Instead of guessing which settings are better, we measure them on the ground truth dataset.

So far we've boosted question to 3.0. The idea was that a query should match the FAQ question. That kind of match should count for more than matching the answer text. It sounds reasonable. But it's a guess, and now we can check it against data instead of trusting it.

This is the main benefit of offline evaluation. We change one parameter, run the same questions again, and see whether the metric moves. The dataset stays fixed, so the comparison is fair.

#### Trying different boosts

Start with a search function where the question boost is configurable:

In [10]:
 q = ground_truth[0]

In [23]:
def search_boost(query, title):
    
    boost_dict = {'synopsis': 10, 'title': title, 'title_english': 0.5, 'studios': 0, 'genres': 1.0, 'source': 1.0}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Evaluate several boost values:

In [24]:
for boost in [0, 0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/2500 [00:00<?, ?it/s]

boost=0: {'hit_rate': 0.754, 'mrr': 0.5989733333333335}


  0%|          | 0/2500 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.7572, 'mrr': 0.59666}


  0%|          | 0/2500 [00:00<?, ?it/s]

...

boost=3.0: {'hit_rate': 0.6508, 'mrr': 0.44444000000000145}


  0%|          | 0/2500 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.4936, 'mrr': 0.28001333333333406}


  0%|          | 0/2500 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.2452, 'mrr': 0.12465999999999976}


For this data we prepared this gives:



The best value here is 0.5 for title_english hit_rate 0.541, mmr 0.370

The best value here is 10 for synopsis hit_rate 0.746, mmr 0.583

The best value here is 0 for studios hit_rate 0.748, mmr 0.586 

The best value here is 1.0 for genres hit_rate 0.748, mmr 0.586 

The best value here is 1.0 for sources hit_rate 0.748, mmr 0.586 

The best value here is 0.5 for sources hit_rate 0.596, mmr 0.586 

I found that fine tuning the boots gave like a 50% increment in hit rate and mmr

Define a search function with all three boosts:

In [37]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Now do a small grid search:

In [38]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/515 [00:00<?, ?it/s]

Sort by MRR:

In [39]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.959223,0.874887
6,1.0,4.0,0.1,0.965049,0.873851
8,1.0,4.0,0.5,0.961165,0.873786
23,2.0,10.0,0.5,0.953398,0.870874
22,2.0,10.0,0.2,0.955340,0.868511
35,5.0,10.0,0.5,0.961165,0.868317
19,2.0,4.0,0.2,0.961165,0.868317
3,1.0,2.0,0.1,0.961165,0.868317
21,2.0,10.0,0.1,0.953398,0.868285
4,1.0,2.0,0.2,0.963107,0.867346


df_results displays the best rows

The best combination weights answer twice as heavily as question, with almost no weight on section. 

So the data says the opposite of where we started. 

The answer text matters more for retrieval than the question text. 

The intuition was wrong, and we'd never have known without measuring it. 

This is exactly why we evaluate instead of guess.

The first three rows have the same relative weights:

question : answer : section = 0.2 : 0.1 : 0.5

So we can use the smaller and easier-to-read values: question=1.0, answer=4.0, and section=0.2. 

This gives the same relative weights as question=5.0, answer=10.0, and section=0.5, 

but the numbers are not unnecessarily large.

Define the search function with these boosts:

In [40]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Usually we care about both metrics. Hit Rate tells us whether the correct document appears at all. 

MRR tells us whether it appears near the top. A document near the top is more likely to be used by the RAG prompt.

### Tuning Workflow

Search parameters can look arbitrary. This includes field boosts, number of results, filters, and other settings. 

Evaluation gives us a way to compare settings with evidence.

Grid search is fine when there are only a few settings. For a larger parameter space, use a smarter search strategy. You can sample random combinations, use Bayesian optimization, or keep a validation split so you don't overfit the evaluation set.

For text search on our dataset, grid search takes about one second per combination. That makes it practical to try many options. 

When each evaluation takes minutes instead of seconds, grid search becomes too expensive. In those cases, use Bayesian optimization with a library like hyperopt. It explores the parameter space more efficiently by focusing on combinations that are likely to improve the metric.

### Top-K tradeoffs

We return 5 results from search. Increasing top-K to 10 would improve hit rate because there are more chances to find the correct document. 

But more results means more context sent to the LLM. That costs more and makes it harder for the model to identify what is relevant. 

Five results is a reasonable default for short FAQ-style documents.

Next, we'll move from retrieval quality to answer quality and evaluate the full RAG pipeline.